#Oxford-Man Data Preparation

This notebook prepares the Oxford-Man Realized Volatility dataset used throughout the thesis.

The workflow includes:

- loading the raw dataset
- selecting indices with sufficient observations
- choosing six representative equity indices
- constructing log-volatility
- building HAR-RV features
- creating one-day-ahead forecasting targets
- saving clean per-index datasets for subsequent modelling

In [2]:
# ============================================================
# OXFORD-MAN DATA PREPARATION
# Master's Thesis: From Fractional to Rough Volatility
# Author: Elisa Battista
# ============================================================

import os
import numpy as np
import pandas as pd

from google.colab import drive

# ------------------------------------------------------------
# Google Drive Setup
# ------------------------------------------------------------

MOUNTPOINT = "/content/drive"

if not os.path.isdir(f"{MOUNTPOINT}/MyDrive"):
    drive.mount(MOUNTPOINT)

BASE = f"{MOUNTPOINT}/MyDrive/thesis"
DATA_PATH = f"{MOUNTPOINT}/MyDrive/realvar.xlsx"

CLEAN_DIR = f"{BASE}/data/clean"

os.makedirs(BASE, exist_ok=True)
os.makedirs(CLEAN_DIR, exist_ok=True)

# ------------------------------------------------------------
# Load Raw Oxford-Man Realized Volatility Dataset
# ------------------------------------------------------------

df = pd.read_excel(DATA_PATH)

print("Head of the raw dataset:")
display(df.head())

print("Raw dataset shape:", df.shape)

required_columns = ["Date", "Symbol", "rv5_ss"]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# ------------------------------------------------------------
# Count Observations per Index
# ------------------------------------------------------------

counts = (
    df["Symbol"]
    .value_counts()
    .reset_index()
)

counts.columns = ["Symbol", "Num_observations"]
counts["Num_observations"] = pd.to_numeric(
    counts["Num_observations"],
    errors="coerce"
)

counts_filtered = counts[counts["Num_observations"] >= 4000]

print("\nIndices with at least 4000 observations:")
display(counts_filtered)

print(f"Total number of indices kept: {len(counts_filtered)}")

# ------------------------------------------------------------
# Filter Dataset
# ------------------------------------------------------------

df_filtered = df[
    df["Symbol"].isin(counts_filtered["Symbol"])
].copy()

print("Filtered dataset shape:", df_filtered.shape)

# ------------------------------------------------------------
# Select Six Representative Equity Indices
# ------------------------------------------------------------

selected_indices = [
    ".STOXX50E",  # EURO STOXX 50
    ".FTSE",     # FTSE 100
    ".SPX",      # S&P 500
    ".IXIC",     # NASDAQ Composite
    ".N225",     # Nikkei 225
    ".HSI"       # Hang Seng Index
]

df_selected = df_filtered[
    df_filtered["Symbol"].isin(selected_indices)
].copy()

df_selected["Date"] = pd.to_datetime(
    df_selected["Date"],
    errors="coerce"
)

df_selected = (
    df_selected
    .sort_values(["Symbol", "Date"])
    .drop_duplicates()
    .reset_index(drop=True)
)

print("\nFinal selected indices:")
print(sorted(df_selected["Symbol"].unique()))

print("Selected dataset shape:", df_selected.shape)

print("\nNumber of observations per selected index:")
display(df_selected.groupby("Symbol").size())

# ------------------------------------------------------------
# Save Reduced Multi-Index Dataset
# ------------------------------------------------------------

REDUCED_PATH = f"{BASE}/realvar_selected.csv"

df_selected.to_csv(
    REDUCED_PATH,
    index=False
)

print(f"\nReduced dataset saved to: {REDUCED_PATH}")

# ------------------------------------------------------------
# Construct Log-Volatility and HAR-RV Features
# ------------------------------------------------------------

df_selected["log_vol"] = 0.5 * np.log(df_selected["rv5_ss"])

grp = df_selected.groupby("Symbol")["log_vol"]

# Daily component: current log-volatility
df_selected["log_vol_d"] = grp.shift(0)

# Weekly component: 5-day rolling average
df_selected["log_vol_w"] = (
    grp.rolling(window=5)
    .mean()
    .reset_index(level=0, drop=True)
)

# Monthly component: 22-day rolling average
df_selected["log_vol_m"] = (
    grp.rolling(window=22)
    .mean()
    .reset_index(level=0, drop=True)
)

# One-day-ahead target
df_selected["target"] = grp.shift(-1)

before = len(df_selected)

df_clean = (
    df_selected
    .dropna()
    .reset_index(drop=True)
)

after = len(df_clean)

print(f"\nRows before dropna: {before}")
print(f"Rows after dropna:  {after}")
print(f"Rows removed:       {before - after}")

# ------------------------------------------------------------
# Save One Clean CSV per Index
# ------------------------------------------------------------

for symbol in selected_indices:
    df_i = df_clean[df_clean["Symbol"] == symbol].copy()

    if df_i.empty:
        print(f"Warning: no data found for {symbol}")
        continue

    output_path = f"{CLEAN_DIR}/data_clean_{symbol}.csv"

    df_i.to_csv(
        output_path,
        index=False
    )

    print(f"Saved {symbol}: {output_path} | rows: {len(df_i)}")

print("\nAll clean per-index datasets saved in:", CLEAN_DIR)
print("Data preparation completed successfully.")

Mounted at /content/drive
Head of the raw dataset:


,Date,Symbol,open_time,close_price,rv5_ss,bv,rk_parzen,rk_twoscale,bv_ss,open_price,nobs,rk_th2,medrv,rv10_ss,rsv_ss,open_to_close,rv10,rsv,rv5,close_time
0,2000-01-03,.AEX,90101,675.44,0.000130,0.000100,0.000179,0.000103,0.000100,675.67,1795,0.000102,0.000050,0.000178,0.000046,-0.000340,0.000178,0.000046,0.000130,163015
1,2000-01-04,.AEX,90416,642.25,0.000201,0.000207,0.000423,0.000199,0.000207,664.20,1785,0.000201,0.000075,0.000261,0.000147,-0.033606,0.000261,0.000147,0.000201,163016
2,2000-01-05,.AEX,90016,632.31,0.000491,0.000361,0.000324,0.000325,0.000361,633.37,1801,0.000345,0.000166,0.000714,0.000328,-0.001675,0.000714,0.000328,0.000491,163016
3,2000-01-06,.AEX,90016,624.21,0.000225,0.000258,0.000219,0.000218,0.000258,632.46,1799,0.000221,0.000152,0.000182,0.000116,-0.013130,0.000182,0.000116,0.000225,163002
4,2000-01-07,.AEX,90046,644.86,0.000138,0.000130,0.000155,0.000126,0.000130,628.93,1798,0.000123,0.000039,0.000157,0.000048,0.025013,0.000157,0.000048,0.000138,163016


Raw dataset shape: (129209, 20)

Indices with at least 4000 observations:


,Symbol,Num_observations
0,.AEX,4714
1,.FCHI,4713
2,.STOXX50E,4713
3,.BFX,4712
4,.GDAXI,4692
5,.IBEX,4681
6,.AORD,4665
7,.FTSE,4660
8,.SPX,4641
9,.MXX,4640


Total number of indices kept: 24
Filtered dataset shape: (109881, 20)

Final selected indices:
['.FTSE', '.HSI', '.IXIC', '.N225', '.SPX', '.STOXX50E']
Selected dataset shape: (27684, 20)

Number of observations per selected index:


,0
Symbol,
.FTSE,4660
.HSI,4532
.IXIC,4637
.N225,4501
.SPX,4641
.STOXX50E,4713



Reduced dataset saved to: /content/drive/MyDrive/thesis/realvar_selected.csv

Rows before dropna: 27684
Rows after dropna:  27552
Rows removed:       132
Saved .STOXX50E: /content/drive/MyDrive/thesis/data/clean/data_clean_.STOXX50E.csv | rows: 4691
Saved .FTSE: /content/drive/MyDrive/thesis/data/clean/data_clean_.FTSE.csv | rows: 4638
Saved .SPX: /content/drive/MyDrive/thesis/data/clean/data_clean_.SPX.csv | rows: 4619
Saved .IXIC: /content/drive/MyDrive/thesis/data/clean/data_clean_.IXIC.csv | rows: 4615
Saved .N225: /content/drive/MyDrive/thesis/data/clean/data_clean_.N225.csv | rows: 4479
Saved .HSI: /content/drive/MyDrive/thesis/data/clean/data_clean_.HSI.csv | rows: 4510

All clean per-index datasets saved in: /content/drive/MyDrive/thesis/data/clean
Data preparation completed successfully.
